## Combine LMP Data and Save Yearly Averages                                                                                                                                                 
                                                                                                                                                                                            
Merges the two raw parquet files (historical and 2-year standard) into a single combined parquet, then computes yearly average LMP by pnode.                                                 
                                                                                                                                                                                            
Inputs:                                                                                                                                                                                      
- `data/raw/pjm_lmp_data/pjm_lmp_historical_raw.parquet` (January 2015 – April 2024)
- `data/raw/pjm_lmp_data/pjm_lmp_2yr_raw.parquet` (May 2024 – March 2026)                                                                                                                    
                                                                                                                                                                                            
Outputs:                                                                                                                                                                                     
- `data/raw/pjm_lmp_data/pjm_lmp_va_all_raw.parquet` — merged full dataset                                                                                                                   
- `data/processed/preprocessing/lmp_data/va_lmp_yearly_avg_all.csv` — yearly averages for all node types (2015–2025)                                                                         
- `data/processed/preprocessing/lmp_data/va_lmp_yearly_avg_load.csv` — yearly averages filtered to LOAD type only 

In [ ]:
# Set up
import pandas as pd
from pathlib import Path
import duckdb

DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

pnode_dir = DATA_DIR / "processed" / "preprocessing" / "pnode_data"
raw_dir = DATA_DIR / "raw" / "pjm_lmp_data"

hist_parquet = raw_dir / "pjm_lmp_historical_raw.parquet"
std_parquet = raw_dir / "pjm_lmp_2yr_raw.parquet"

In [30]:
# Merge both raw parquet files into one for querying
con = duckdb.connect()
                                                                                                                                                                                                
con.execute(f"""
    COPY (                                                                                                                                                                                     
        SELECT * FROM read_parquet('{hist_parquet}')
        UNION ALL
        SELECT * FROM read_parquet('{std_parquet}')                                                                                                                                            
    ) TO '{raw_dir}/pjm_lmp_va_all_raw.parquet' (FORMAT PARQUET)
""")                                                                                                                                                                                           
                
con.close()                                                                                                                                                                                    
print("Merged parquet saved")

Merged parquet saved


In [31]:
# Check merged parquet file
con = duckdb.connect()

all_raw = f"{raw_dir}/pjm_lmp_va_all_raw.parquet"

print(con.execute(f"SELECT COUNT(*) FROM read_parquet('{all_raw}')").fetchone())
print(con.execute(f"SELECT MIN(datetime_beginning_ept), MAX(datetime_beginning_ept) FROM read_parquet('{all_raw}')").fetchdf())
print(con.execute(f"SELECT COUNT(DISTINCT pnode_id) FROM read_parquet('{all_raw}')").fetchdf())
print(con.execute(f"SELECT type, COUNT(*) FROM read_parquet('{all_raw}') GROUP BY type").fetchdf())

con.close()

(99490765,)
  min(datetime_beginning_ept) max(datetime_beginning_ept)
0                    2015-01-                    2026-03-
   count(DISTINCT pnode_id)
0                      1139
   type  count_star()
0   EHV       3500128
1   GEN      17022412
2  LOAD      78968225


In [ ]:
# Export yearly averages to CSV for years 2015-2025
con = duckdb.connect()

all_raw = f"{raw_dir}/pjm_lmp_va_all_raw.parquet"
processed_dir = DATA_DIR / "processed" / "preprocessing" / "lmp_data" 

yearly = con.execute(f"""
    SELECT 
        substr(datetime_beginning_ept, 1, 4)::INT AS year,
        pnode_id,
        pnode_name,
        type,
        AVG(total_lmp_rt) AS avg_total_lmp,
        AVG(system_energy_price_rt) AS avg_energy,
        AVG(congestion_price_rt) AS avg_congestion,
        AVG(marginal_loss_price_rt) AS avg_loss,
        COUNT(*) AS hours
    FROM read_parquet('{all_raw}')
    WHERE substr(datetime_beginning_ept, 1, 4)::INT BETWEEN 2015 AND 2025
    GROUP BY year, pnode_id, pnode_name, type
    ORDER BY year, pnode_id, type
""").fetchdf()

con.close()

print(f"Rows: {len(yearly):,}")
print(f"Year range: {yearly['year'].min()} – {yearly['year'].max()}")
print(f"Unique pnode IDs: {yearly['pnode_id'].nunique():,}")

yearly.to_csv(processed_dir / "va_lmp_yearly_avg_all.csv", index=False)
print("Saved va_lmp_yearly_avg_all.csv")

Rows: 11,257
Year range: 2015 – 2025
Unique pnode IDs: 1,113
Saved va_lmp_yearly_avg_all.csv


In [33]:
# Create df and inspect
lmp = pd.read_csv(processed_dir / "va_lmp_yearly_avg_all.csv")

print("Shape:", lmp.shape)
print("\nYears:", sorted(lmp["year"].unique()))
print("\nTypes:", lmp["type"].value_counts().to_dict())
print("\nUnique pnodes:", lmp["pnode_id"].nunique())
print("\nMissing values:\n", lmp.isnull().sum())
print("\nSummary statistics:")
lmp[["avg_total_lmp", "avg_energy", "avg_congestion", "avg_loss", "hours"]].describe().round(3)

Shape: (11257, 9)

Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Types: {'LOAD': 8886, 'GEN': 1931, 'EHV': 440}

Unique pnodes: 1113

Missing values:
 year              0
pnode_id          0
pnode_name        0
type              0
avg_total_lmp     0
avg_energy        0
avg_congestion    0
avg_loss          0
hours             0
dtype: int64

Summary statistics:


,avg_total_lmp,avg_energy,avg_congestion,avg_loss,hours
count,11257.000,11257.000,11257.000,11257.000,11257.000
mean,39.240,35.890,3.022,0.329,8623.496
std,16.508,13.885,4.847,0.728,969.106
min,16.558,20.626,-11.067,-2.971,72.000
25%,29.817,27.525,1.192,-0.067,8760.000
50%,34.300,31.230,2.391,0.198,8760.000
75%,41.195,38.125,3.503,0.583,8784.000
max,157.215,137.565,55.708,9.241,8784.000


The table va_lmp_yearly_avg_all.csv contains one row per pricing node, per year, per node type. It shows the average hourly electricity price at that node across the entire year.
Specifically, for each pnode id, for each year from 2015–2025, it records:

- What the node is — its ID, name, and type (LOAD, GEN, EHV)
- What electricity cost on average — the total LMP (locational marginal price), broken into its three components:
- Energy — the baseline system-wide price, same for everyone
- Congestion — extra cost caused by transmission constraints at that specific location
- Marginal loss — extra cost from electrical losses in the transmission lines
- How many hours of data went into that average

So it answers the question: "At this specific pricing node in Virginia, what did electricity cost on average in this year, and how much of that cost was due to inefficiencies (congestion + losses)?"

In [34]:
# Filter only to LOAD types for analysis
lmp_load = yearly[yearly["type"] == "LOAD"]
lmp_load.to_csv(processed_dir / "va_lmp_yearly_avg_load.csv", index=False)
print(f"Saved va_lmp_yearly_avg_load.csv ({len(lmp_load):,} rows)")

Saved va_lmp_yearly_avg_load.csv (8,886 rows)


We notice in the previous summaries that the raw lmp data has 1139 unique pnode ids, but the aggregated data only has 1113. Why? Let's check. 

In [ ]:
# Find which pnodes are missing from LMP data    
# Load original pnode list first                                                                                                                                                            
pnodes = pd.read_csv(pnode_dir / "va_pnode_ids_final_full.csv") 
print(pnodes.columns.tolist())                                                                                                                                       
lmp_pnodes = set(lmp["pnode_id"].unique())   
original_pnodes = set(pnodes[pnodes["type"] != "AGGREGATE"]["pnode_id"].unique())                                                                                                                                   
                                                                                                                                                                                        
missing = original_pnodes - lmp_pnodes
print(f"Missing pnodes: {len(missing)}")                                                                                                                                               
                                        
# Look them up in original list                                                                                                                                                        
missing_info = pnodes[pnodes["pnode_id"].isin(missing)]
print(missing_info[["pnode_id", "pnode_match", "type"]].to_string())  

['substation_name', 'latitude', 'longitude', 'pnode_match', 'pnode_id', 'type']
Missing pnodes: 26
        pnode_id pnode_match  type
54    2156118955    BEAUMEAD  LOAD
55    2156119644    BEAUMEAD  LOAD
60    2156119639        BECO  LOAD
98    2156119333      BREMO4  LOAD
217   2156119116      CLOVER  LOAD
218   2156119223      CLOVER  LOAD
219   2156119319      CLOVER  LOAD
220   2156119345      CLOVER  LOAD
263   2156119179    DAVISDRV  LOAD
264   2156119430    DAVISDRV  LOAD
708   2156119470        NIVO  LOAD
834   2156118954    POSSUMP4  LOAD
835   2156119173    POSSUMP4  LOAD
836   2156119191    POSSUMP4  LOAD
837   2156119312    POSSUMP4  LOAD
838   2156119516    POSSUMP4  LOAD
839   2156119591    POSSUMP4  LOAD
840   2156119598    POSSUMP4  LOAD
943   2156119068    SHELHORN  LOAD
944   2156119087    SHELHORN  LOAD
1052  2156119492       TREGO  LOAD
1102  2156119243    WATTSVIL   GEN
1148  2156119284    YORKTOW4  LOAD
1149  2156119462    YORKTOW4  LOAD
1150  2156119546    YORKTO

In [36]:
# Check if these are from 2026
con = duckdb.connect()

all_raw = f"{raw_dir}/pjm_lmp_va_all_raw.parquet"

result = con.execute(f"""
    SELECT pnode_id, substr(datetime_beginning_ept, 1, 4)::INT AS year, COUNT(*) AS rows
    FROM read_parquet('{all_raw}')
    WHERE pnode_id IN ({','.join(str(p) for p in missing)})
    GROUP BY pnode_id, year
    ORDER BY pnode_id, year
""").fetchdf()

con.close()
print(result.to_string())

      pnode_id  year  rows
0   2156118954  2026   504
1   2156118955  2026   504
2   2156119068  2026   504
3   2156119087  2026   504
4   2156119116  2026   504
5   2156119173  2026   504
6   2156119179  2026   504
7   2156119191  2026   504
8   2156119223  2026   504
9   2156119243  2026   504
10  2156119284  2026   504
11  2156119312  2026   504
12  2156119319  2026   504
13  2156119333  2026   504
14  2156119345  2026   504
15  2156119430  2026   504
16  2156119462  2026   504
17  2156119470  2026   504
18  2156119492  2026   504
19  2156119516  2026   504
20  2156119546  2026   504
21  2156119591  2026   504
22  2156119598  2026   504
23  2156119606  2026   504
24  2156119639  2026   504
25  2156119644  2026   504


Answer: these pnode ids were introduced in 2026, which is not included in the aggregate data for the analysis. 